# Pipeline de datos: accidentalidad vial en Colombia
**Gestión de Datos Masivos – G002 | Institución Universitaria Pascual Bravo**

Este notebook implementa el recorrido **API → Raw → Kafka → HDFS/MapReduce → PySpark/Parquet → calidad e idempotencia → streaming → SQLite/tablero**.

## Fuente seleccionada
**Vehículos involucrados en un accidente de tránsito – Ley 2251 de 2022**, publicado en datos.gov.co por el Ministerio de Transporte / RUNT 2.0. El catálogo consultado reporta aproximadamente **450.000 filas y 9 columnas**. Incluye marca, modelo, tipo y edad del vehículo, fecha del siniestro, gravedad, departamento, municipio y autoridad de tránsito.

- Dataset Socrata: `6jmc-vaxk`
- API: `https://www.datos.gov.co/resource/6jmc-vaxk.json`
- Fuente: [Portal Nacional de Datos Abiertos](https://www.datos.gov.co/Transporte/VEHICULOS-INVOLUCRADOS-EN-UN-ACCIDENTE-DE-TRANSITO/6jmc-vaxk)

**Pregunta principal:** ¿Qué patrones se observan en los vehículos involucrados en siniestros viales con heridos o muertos en Colombia, según tipo de vehículo, gravedad, antigüedad y ubicación?

**Usuarios de los indicadores:** autoridades de tránsito, Ministerio de Transporte, alcaldías, secretarías de movilidad e investigadores de seguridad vial.

> Alcance y limitación: el conjunto describe vehículos involucrados en siniestros con resultado de muerte o lesión; no representa necesariamente todos los accidentes de tránsito del país ni permite calcular tasas de riesgo sin denominadores de exposición (por ejemplo, parque automotor o kilómetros recorridos).

## Arquitectura propuesta: Lambda
Se plantea **Lambda** porque el análisis histórico se procesa por lotes (raw, HDFS, MapReduce y Spark) y, adicionalmente, se habilita una capa rápida de streaming desde Kafka para observar eventos que llegan durante la ejecución. En el notebook, Kafka/HDFS se activan si existe un broker/cluster disponible; se incluye un modo local reproducible para validar la lógica cuando Colab no tiene esos servicios.

## 0. Preparación del entorno
Ejecute las celdas en orden. En Google Colab, la instalación puede tardar unos minutos. Para Kafka y HDFS se requiere un servicio externo o un entorno Docker; el resto del flujo puede ejecutarse localmente.

In [ ]:
# Instalar dependencias principales (en Colab descomente la línea)
# %pip -q install requests pandas pyarrow pyspark kafka-python sqlalchemy plotly

import os, sys, json, time, hashlib, shutil, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd

BASE_DIR = Path("/content/pipeline_accidentalidad") if Path("/content").exists() else Path.cwd() / "pipeline_accidentalidad"
RAW_DIR = BASE_DIR / "raw"
MANIFEST_DIR = BASE_DIR / "manifests"
LOCAL_LAKE_DIR = BASE_DIR / "lake"
CURATED_DIR = BASE_DIR / "curated"
REPORT_DIR = BASE_DIR / "reports"
SERVICE_DIR = BASE_DIR / "service"
for p in [RAW_DIR, MANIFEST_DIR, LOCAL_LAKE_DIR, CURATED_DIR, REPORT_DIR, SERVICE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_ID = "6jmc-vaxk"
API_URL = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"
SOURCE_NAME = "datos.gov.co"
DATASET_NAME = "vehiculos_involucrados_accidentes_transito"
PAGE_SIZE = 50000
INGEST_DATE = datetime.now().strftime("%Y-%m-%d")
print("Directorio de trabajo:", BASE_DIR)
print("API:", API_URL)

## 1. Ingesta a zona Raw: API, paginación, reintentos y manifiesto

Cada fila original se conserva sin modificar dentro de `payload`. El sobre agrega fuente, dataset, identificador, fecha de ingesta y fecha de evento cuando se puede interpretar.

El dataset no publica en el catálogo un identificador único de fila. Por eso se construye `id_registro` como SHA-256 del contenido original normalizado y ordenado. Esta clave permite reconocer filas idénticas; si dos eventos distintos tuvieran exactamente los mismos nueve atributos, quedarían indistinguibles. Esta limitación debe mencionarse en la sustentación.

In [ ]:
def get_with_retries(url, params, max_retries=5, timeout=90):
    last_error = None
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_error = exc
            wait = min(2 ** attempt, 30)
            print(f"Error API (intento {attempt+1}/{max_retries}): {exc}. Reintento en {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"No fue posible consultar la API: {last_error}")

def stable_record_id(record):
    canonical = json.dumps(record, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def ingest_api(max_records=None, page_size=PAGE_SIZE):
    run_ts = datetime.now(timezone.utc).isoformat()
    ingest_date = datetime.now().strftime("%Y-%m-%d")
    output_file = RAW_DIR / f"fecha_ingesta={ingest_date}" / "lote_0001.jsonl"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    offset, total = 0, 0
    params_base = {"$order": "fecha_accidente", "$limit": page_size}
    with output_file.open("a", encoding="utf-8") as f:
        while True:
            params = dict(params_base, **{"$offset": offset})
            if max_records is not None:
                remaining = max_records - total
                if remaining <= 0:
                    break
                params["$limit"] = min(page_size, remaining)
            response = get_with_retries(API_URL, params)
            batch = response.json()
            if not batch:
                break
            for record in batch:
                envelope = {
                    "fuente": SOURCE_NAME,
                    "dataset": DATASET_NAME,
                    "id_registro": stable_record_id(record),
                    "fecha_ingesta": ingest_date,
                    "fecha_evento": record.get("fecha_accidente"),
                    "payload": record
                }
                f.write(json.dumps(envelope, ensure_ascii=False) + "\n")
            total += len(batch)
            offset += len(batch)
            print(f"Descargados: {total:,}")
            if len(batch) < params["$limit"]:
                break

    manifest = {
        "run_timestamp_utc": run_ts,
        "url": API_URL,
        "parametros": params_base,
        "registros_descargados": total,
        "archivo_raw": str(output_file),
        "fecha_ingesta": ingest_date,
        "dataset_id": DATASET_ID
    }
    manifest_path = MANIFEST_DIR / f"manifest_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return manifest, output_file

# Primera ejecución: None descarga todas las filas; para una prueba rápida use max_records=5000.
# Si se vuelve a ejecutar, el archivo se abre en modo append; la etapa curada elimina duplicados.
manifest, raw_file = ingest_api(max_records=None)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

### Inspección rápida de Raw

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

raw_records = read_jsonl(raw_file)
print("Registros leídos:", len(raw_records))
if raw_records:
    print("Campos del payload:", list(raw_records[0]["payload"].keys()))
    print("Ejemplo de sobre:")
    print(json.dumps(raw_records[0], ensure_ascii=False, indent=2)[:1800])

## 2. Kafka: tópico de 3 particiones, productor y consumidor

Para ejecutar esta fase debe existir Kafka accesible. Configure `KAFKA_BOOTSTRAP_SERVERS` (por ejemplo `localhost:9092`) y cree el tópico `accidentalidad-raw` con tres particiones. La clave del mensaje es `id_registro`, lo que conserva la afinidad de una misma clave dentro de una partición.

En Colab sin broker, la celda informa que la fase queda pendiente y permite continuar con el modo local. Para evidencias completas, ejecutar Kafka mediante Docker Compose o en una máquina con Docker.

In [ ]:
KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
KAFKA_TOPIC = "accidentalidad-raw"
KAFKA_GROUP = "grupo-accidentalidad"

def kafka_publish_and_consume(path, max_messages=None):
    try:
        from kafka import KafkaProducer, KafkaConsumer
        from kafka.admin import KafkaAdminClient, NewTopic
        from kafka.errors import TopicAlreadyExistsError
    except ImportError:
        print("Falta kafka-python. Instale con: pip install kafka-python")
        return None

    admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP)
    try:
        admin.create_topics([NewTopic(name=KAFKA_TOPIC, num_partitions=3, replication_factor=1)])
        print("Tópico creado con 3 particiones.")
    except Exception as e:
        print("El tópico puede existir previamente:", e)
    admin.close()

    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        key_serializer=lambda k: k.encode("utf-8"),
        value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode("utf-8"),
        acks="all"
    )
    sent = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            env = json.loads(line)
            producer.send(KAFKA_TOPIC, key=env["id_registro"], value=env)
            sent += 1
            if max_messages and sent >= max_messages:
                break
    producer.flush()
    producer.close()

    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=KAFKA_BOOTSTRAP,
        group_id=KAFKA_GROUP,
        auto_offset_reset="earliest",
        enable_auto_commit=True,
        consumer_timeout_ms=10000,
        key_deserializer=lambda k: k.decode("utf-8") if k else None,
        value_deserializer=lambda v: json.loads(v.decode("utf-8"))
    )
    received = []
    for msg in consumer:
        received.append(msg.value)
        if max_messages and len(received) >= max_messages:
            break
    consumer.close()
    print(f"Mensajes publicados: {sent}; mensajes leídos en esta sesión: {len(received)}")
    return received

# Requiere Kafka activo. Para no publicar cientos de miles de mensajes durante pruebas,
# puede limitarse a 5000 y luego repetir con None para la entrega.
kafka_messages = kafka_publish_and_consume(raw_file, max_messages=5000)

**Duplicados en Kafka:** si el productor se ejecuta dos veces, Kafka recibe dos publicaciones, aunque tengan la misma key. La key no elimina duplicados por sí sola. Se controlan en el consumidor/zona curada mediante `id_registro` y una escritura idempotente (upsert o deduplicación). En producción también se puede usar productor idempotente y control de offsets.

## 3. Data lake HDFS y MapReduce

La estructura objetivo es `fuente/dataset/fecha_ingesta=AAAA-MM-DD/lote.jsonl`. Se intenta escribir en HDFS si `HDFS_NAMENODE` está configurado y el cliente `hdfs dfs` está disponible. Si no, se crea una copia local con la misma estructura para poder ejecutar las siguientes fases.

MapReduce se implementa con funciones mapper/reducer de Python sobre los datos Raw. El cálculo será el conteo de vehículos por gravedad del siniestro.

In [ ]:
def save_batch_to_lake(raw_path):
    local_target = LOCAL_LAKE_DIR / SOURCE_NAME / DATASET_NAME / f"fecha_ingesta={INGEST_DATE}" / "lote_0001.jsonl"
    local_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(raw_path, local_target)

    namenode = os.getenv("HDFS_NAMENODE")
    hdfs_target = f"{namenode.rstrip('/')}/datos/{SOURCE_NAME}/{DATASET_NAME}/fecha_ingesta={INGEST_DATE}/lote_0001.jsonl" if namenode else None
    if hdfs_target and shutil.which("hdfs"):
        subprocess.run(["hdfs", "dfs", "-mkdir", "-p", str(Path(hdfs_target).parent)], check=True)
        subprocess.run(["hdfs", "dfs", "-put", "-f", str(raw_path), hdfs_target], check=True)
        print("Lote guardado en HDFS:", hdfs_target)
    else:
        print("HDFS no está configurado; se guardó copia local:", local_target)
    return local_target

lake_file = save_batch_to_lake(raw_file)

def mapper_gravedad(envelope):
    payload = envelope.get("payload", {})
    gravedad = str(payload.get("gravedad_accidente") or "SIN_DATO").strip().upper()
    yield gravedad, 1

def reducer_count(mapped_items):
    totals = {}
    for key, value in mapped_items:
        totals[key] = totals.get(key, 0) + value
    return totals

mapped = []
for env in read_jsonl(lake_file):
    mapped.extend(mapper_gravedad(env))
mapreduce_result = reducer_count(mapped)
print("Conteo MapReduce por gravedad:")
print(pd.DataFrame(mapreduce_result.items(), columns=["gravedad_accidente", "vehiculos"]))

## 4. Zona curada con PySpark y Parquet

Se estandarizan nombres de columnas, tipos y valores. Se eliminan duplicados por `id_registro` y se convierten números y fechas de forma tolerante para que los valores mal formados no detengan Spark.

La fuente contiene fechas completas y valores parciales como `01/2022` (mes/año). Para estos últimos se usa el primer día del mes como fecha representativa; los formatos que no se puedan interpretar quedan como `NULL`. Los registros sin departamento o municipio se conservan y se etiquetan como `SIN DATO`, evitando descartarlos por falta de información geográfica.

La partición Parquet es por `anio_accidente`; facilita consultas temporales y evita particionar por columnas de alta cardinalidad como municipio.

In [ ]:
# Instalar PySpark si hace falta:
# %pip -q install pyspark pyarrow

from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.master("local[*]").appName(
    "PipelineAccidentalidadColombia"
).getOrCreate()

# Spark lee los JSONL de los sobres.
df_raw = spark.read.json(str(lake_file))
before_count = df_raw.count()
print("Filas antes de limpieza:", before_count)

# Extraer los campos del payload.
df = df_raw.select(
    "id_registro", "fecha_ingesta",
    F.col("payload.marca_vehiculo").alias("marca_vehiculo"),
    F.col("payload.modelo_vehiculo").alias("modelo_vehiculo"),
    F.col("payload.tipo_vehiculo").alias("tipo_vehiculo"),
    F.col("payload.edad_vehiculo").alias("edad_vehiculo"),
    F.col("payload.fecha_accidente").cast("string").alias("fecha_accidente"),
    F.col("payload.gravedad_accidente").alias("gravedad_accidente"),
    F.col("payload.departamento_accidente").alias("departamento"),
    F.col("payload.municipio_accidente").alias("municipio"),
    F.col("payload.autoridad_de_transito").alias("autoridad_transito")
)

# Normalizar texto sin eliminar los registros que tengan campos descriptivos vacíos.
df = (
    df
    .withColumn("marca_vehiculo", F.initcap(F.trim(F.col("marca_vehiculo"))))
    .withColumn("tipo_vehiculo", F.upper(F.trim(F.col("tipo_vehiculo"))))
    .withColumn("gravedad_accidente", F.initcap(F.trim(F.col("gravedad_accidente"))))
    .withColumn("departamento", F.initcap(F.trim(F.col("departamento"))))
    .withColumn("municipio", F.initcap(F.trim(F.col("municipio"))))
)

# Conversión tolerante de números: los valores inválidos quedan como NULL,
# pero no provocan que Spark detenga el notebook (modo ANSI).
df = (
    df
    .withColumn("modelo_vehiculo", F.expr("try_cast(modelo_vehiculo AS INT)"))
    .withColumn("edad_vehiculo", F.expr("try_cast(edad_vehiculo AS INT)"))
)

# La fuente puede contener fechas completas o solo mes/año, por ejemplo 01/2022.
# Para mes/año se asigna el primer día del mes como fecha representativa.
df = df.withColumn(
    "_fecha_normalizada",
    F.when(
        F.trim(F.col("fecha_accidente")).rlike(r"^\\d{2}/\\d{4}$"),
        F.concat(F.lit("01/"), F.trim(F.col("fecha_accidente")))
    ).otherwise(F.trim(F.col("fecha_accidente")))
)

# try_to_timestamp devuelve NULL para formatos inválidos en lugar de lanzar excepción.
df = df.withColumn(
    "fecha_parseada",
    F.coalesce(
        F.expr("try_to_timestamp(_fecha_normalizada, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(_fecha_normalizada, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(_fecha_normalizada, 'dd/MM/yyyy')"),
        F.expr("try_to_timestamp(_fecha_normalizada, 'dd/MM/yyyy HH:mm:ss')"),
        F.expr("try_to_timestamp(_fecha_normalizada, 'MM/dd/yyyy')"),
        F.expr("try_to_timestamp(_fecha_normalizada, 'MM/dd/yyyy HH:mm:ss')"),
        F.expr("try_to_timestamp(_fecha_normalizada)")
    )
).drop("_fecha_normalizada")

df = (
    df
    .withColumn("anio_accidente", F.year("fecha_parseada"))
    .dropDuplicates(["id_registro"])
)

# Se descartan únicamente filas sin identificador. Las filas sin departamento
# o municipio se conservan y se etiquetan como SIN DATO para no perder información.
df_clean = (
    df
    .filter(F.col("id_registro").isNotNull())
    .withColumn("departamento", F.coalesce(F.col("departamento"), F.lit("SIN DATO")))
    .withColumn("municipio", F.coalesce(F.col("municipio"), F.lit("SIN DATO")))
    .withColumn("tipo_vehiculo", F.coalesce(F.col("tipo_vehiculo"), F.lit("SIN DATO")))
    .withColumn("gravedad_accidente", F.coalesce(F.col("gravedad_accidente"), F.lit("SIN DATO")))
)

after_count = df_clean.count()
print("Filas antes de limpieza:", before_count)
print("Filas después de limpieza:", after_count)
print("Filas eliminadas (principalmente sin id_registro o duplicadas):", before_count - after_count)
print("Fechas que no se pudieron interpretar:",
      df_clean.filter(F.col("fecha_parseada").isNull()).count())
print("Registros sin departamento:",
      df_clean.filter(F.col("departamento") == "SIN DATO").count())
print("Registros sin municipio:",
      df_clean.filter(F.col("municipio") == "SIN DATO").count())

df_clean.show(10, truncate=False)

# Particionar por año; los registros sin fecha interpretable van a DESCONOCIDO.
df_out = df_clean.withColumn(
    "anio_accidente",
    F.coalesce(F.col("anio_accidente").cast("string"), F.lit("DESCONOCIDO"))
)

parquet_path = str(CURATED_DIR / "vehiculos_accidentes_parquet")
df_out.write.mode("overwrite").partitionBy("anio_accidente").parquet(parquet_path)
print("Parquet guardado en:", parquet_path)

### Comparación de tamaño JSONL vs Parquet

In [ ]:
json_size = lake_file.stat().st_size
parquet_size = sum(p.stat().st_size for p in Path(parquet_path).rglob("*") if p.is_file())
print(f"JSONL: {json_size:,} bytes ({json_size/1024/1024:.2f} MB)")
print(f"Parquet: {parquet_size:,} bytes ({parquet_size/1024/1024:.2f} MB)")
if json_size:
    print(f"Reducción de tamaño: {(1 - parquet_size/json_size)*100:.2f}%")

## 5. Calidad, carga incremental e idempotencia

Se ejecutan al menos cuatro controles: volumen mayor que cero, identificadores no nulos, ausencia de duplicados y valores de edad/modelo no negativos cuando existen. Se guarda un reporte con aprobados, fallidos y filas descartadas. Las validaciones críticas detienen el flujo.

La marca de agua se calcula como la mayor fecha de accidente interpretada. La deduplicación por `id_registro` permite volver a procesar el mismo lote sin incrementar la cantidad lógica de registros. Para una operación productiva, se debe persistir el estado de la marca de agua y consultar el origen desde esa fecha, adaptando el filtro al formato real del campo de fecha.

In [ ]:
total_curated = df_clean.count()
null_ids = df_clean.filter(F.col("id_registro").isNull()).count()
duplicate_ids = total_curated - df_clean.select("id_registro").distinct().count()
negative_age = df_clean.filter(F.col("edad_vehiculo") < 0).count()
negative_model = df_clean.filter(F.col("modelo_vehiculo") < 0).count()

quality_checks = [
    {"validacion": "conteo_mayor_cero", "resultado": total_curated > 0, "valor": total_curated, "critica": True},
    {"validacion": "id_registro_no_nulo", "resultado": null_ids == 0, "valor": null_ids, "critica": True},
    {"validacion": "sin_id_duplicados", "resultado": duplicate_ids == 0, "valor": duplicate_ids, "critica": True},
    {"validacion": "edad_y_modelo_no_negativos", "resultado": negative_age == 0 and negative_model == 0,
     "valor": {"edad_negativa": negative_age, "modelo_negativo": negative_model}, "critica": False}
]
quality_report = {
    "fecha_reporte": datetime.now().isoformat(),
    "filas_raw": before_count,
    "filas_curadas": total_curated,
    "filas_descartadas": before_count - total_curated,
    "validaciones": quality_checks
}
quality_path = REPORT_DIR / "reporte_calidad.json"
quality_path.write_text(json.dumps(quality_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(quality_report, ensure_ascii=False, indent=2))

failed_critical = [x for x in quality_checks if x["critica"] and not x["resultado"]]
if failed_critical:
    raise RuntimeError(f"Falló una validación crítica: {failed_critical}")
print("Validaciones críticas superadas.")

In [ ]:
# Demostración de idempotencia: escribir dos veces el mismo conjunto no cambia
# el número de identificadores únicos.
count_run_1 = df_clean.select("id_registro").distinct().count()
df_second_run = df_clean.unionByName(df_clean).dropDuplicates(["id_registro"])
count_run_2 = df_second_run.select("id_registro").distinct().count()
print("Registros únicos después de primera corrida:", count_run_1)
print("Registros únicos después de reprocesar el mismo lote:", count_run_2)
print("¿Idempotente?:", count_run_1 == count_run_2)

watermark = df_clean.agg(F.max("fecha_parseada").alias("ultima_fecha")).first()["ultima_fecha"]
watermark_text = watermark.isoformat() if watermark else None
watermark_file = REPORT_DIR / "watermark.json"
watermark_file.write_text(json.dumps({"ultima_fecha_cargada": watermark_text}, indent=2), encoding="utf-8")
print("Marca de agua:", watermark_text)

## 6. Streaming con Spark Structured Streaming

La implementación real lee desde Kafka, agrupa eventos por ventana de 1 hora, aplica watermark y guarda progreso en checkpoint. Para ejecutarla se necesita un broker Kafka disponible y una versión de Spark con el conector Kafka correspondiente.

La celda no simula que hubo streaming si Kafka no está disponible: informa el requisito y deja el código listo para ejecutar cuando se configure el servicio.

In [ ]:
def run_structured_streaming():
    bootstrap = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
    checkpoint = str(BASE_DIR / "checkpoints" / "accidentalidad")
    output = str(BASE_DIR / "streaming_output")
    stream_df = (spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", bootstrap)
        .option("subscribe", KAFKA_TOPIC)
        .option("startingOffsets", "earliest")
        .load())

    parsed = stream_df.select(
        F.from_json(F.col("value").cast("string"), T.StructType([
            T.StructField("id_registro", T.StringType()),
            T.StructField("fecha_evento", T.StringType()),
            T.StructField("payload", T.MapType(T.StringType(), T.StringType()))
        ])).alias("evento")
    ).select("evento.*")

    events = (parsed
        .withColumn("event_time", F.to_timestamp("fecha_evento"))
        .filter(F.col("event_time").isNotNull())
        .withWatermark("event_time", "2 hours")
        .groupBy(F.window("event_time", "1 hour"))
        .agg(F.count("*").alias("vehiculos_reportados"))
    )

    query = (events.writeStream
        .format("parquet")
        .option("path", output)
        .option("checkpointLocation", checkpoint)
        .outputMode("append")
        .start())
    print("Streaming iniciado. Detenga la consulta con query.stop() cuando termine.")
    return query

# Para iniciar: streaming_query = run_structured_streaming()
# Para detener: streaming_query.stop()

**Watermark y checkpoint:** el watermark permite manejar eventos tardíos dentro del margen de dos horas; los eventos más antiguos que el umbral pueden quedar fuera de agregaciones ya cerradas. El checkpoint conserva offsets y estado para recuperación después de reiniciar. Para demostrarlo en la sustentación, inicie el flujo, publique mensajes, deténgalo y vuelva a iniciarlo usando la misma ruta de checkpoint; compare los conteos y evidencie que no se duplican.

## 7. Zona de servicio: indicadores en SQLite
Se preparan indicadores agregados para que el tablero consuma tablas listas, sin tener que hacer cálculos pesados. Los indicadores incluyen:
1. Vehículos involucrados por tipo.
2. Vehículos involucrados por departamento.
3. Vehículos involucrados por gravedad.
4. Antigüedad promedio del vehículo por tipo (si la variable tiene datos válidos).

In [ ]:
from sqlalchemy import create_engine
engine = create_engine(f"sqlite:///{SERVICE_DIR / 'accidentalidad.db'}")

# Agregaciones Spark -> Pandas -> SQLite
by_type = (df_clean.groupBy("tipo_vehiculo").agg(F.count("*").alias("cantidad_vehiculos"))
           .orderBy(F.desc("cantidad_vehiculos")).toPandas())
by_department = (df_clean.groupBy("departamento").agg(F.count("*").alias("cantidad_vehiculos"))
                 .orderBy(F.desc("cantidad_vehiculos")).toPandas())
by_severity = (df_clean.groupBy("gravedad_accidente").agg(F.count("*").alias("cantidad_vehiculos"))
               .orderBy(F.desc("cantidad_vehiculos")).toPandas())
avg_age = (df_clean.groupBy("tipo_vehiculo").agg(F.avg("edad_vehiculo").alias("edad_promedio"))
           .orderBy(F.desc("edad_promedio")).toPandas())

by_type.to_sql("vehiculos_por_tipo", engine, if_exists="replace", index=False)
by_department.to_sql("vehiculos_por_departamento", engine, if_exists="replace", index=False)
by_severity.to_sql("vehiculos_por_gravedad", engine, if_exists="replace", index=False)
avg_age.to_sql("edad_promedio_por_tipo", engine, if_exists="replace", index=False)

print("Tablas de servicio cargadas en SQLite.")
display(by_type.head(15))
display(by_department.head(15))
display(by_severity)

## 8. Tablero con al menos tres visualizaciones
El tablero se construye con Plotly y muestra la fuente y fecha de corte. Las cifras representan vehículos involucrados reportados en el conjunto, no el total de accidentes ni tasas de accidentalidad.

In [ ]:
# %pip -q install plotly
import plotly.express as px
from IPython.display import display, HTML

cutoff = datetime.now().strftime("%Y-%m-%d")
fig1 = px.bar(by_type.head(12), x="tipo_vehiculo", y="cantidad_vehiculos",
              title="Vehículos involucrados por tipo", text_auto=True)
fig2 = px.bar(by_department.head(15), x="departamento", y="cantidad_vehiculos",
              title="Departamentos con más vehículos involucrados", text_auto=True)
fig3 = px.bar(by_severity, x="gravedad_accidente", y="cantidad_vehiculos",
              title="Vehículos según gravedad del siniestro", text_auto=True)
fig4 = px.bar(avg_age.dropna().head(12), x="tipo_vehiculo", y="edad_promedio",
              title="Edad promedio del vehículo por tipo", text_auto=".1f")

for fig in [fig1, fig2, fig3, fig4]:
    fig.update_layout(xaxis_title="", yaxis_title="Cantidad / años", template="plotly_white")
    fig.show()

print(f"Fuente: datos.gov.co – dataset {DATASET_ID} | Fecha de corte de ejecución: {cutoff}")

## 9. Evidencias y resumen de ejecución

Guarde capturas de: tópico Kafka con tres particiones, productor/consumidor, `hdfs dfs -ls`, comparación MapReduce/Spark, conteos de idempotencia, reporte de calidad, reinicio de streaming y tablero. En Colab, descargue la carpeta de resultados o súbala al repositorio.

### Comparación MapReduce vs Spark

In [ ]:
# Comparar ambos cálculos sobre el mismo Raw, con la misma regla de normalización.
spark_raw_severity = (df_raw
    .withColumn("gravedad_norm", F.upper(F.trim(F.coalesce(F.col("payload.gravedad_accidente"), F.lit("SIN_DATO")))))
    .groupBy("gravedad_norm").count())
spark_severity = {row["gravedad_norm"]: row["count"] for row in spark_raw_severity.collect()}

print("MapReduce:", mapreduce_result)
print("Spark sobre Raw:", spark_severity)
print("¿Coinciden?:", mapreduce_result == spark_severity)


## Conclusiones
- El pipeline integra una fuente pública real mediante API y conserva la trazabilidad de cada registro.
- La zona curada facilita análisis reproducibles al estandarizar campos y deduplicar por identificador derivado.
- Parquet optimiza el almacenamiento y la consulta analítica frente a JSONL.
- Los indicadores permiten identificar concentraciones por tipo de vehículo, gravedad y territorio.
- Kafka, HDFS y streaming requieren servicios activos; su ejecución debe evidenciarse en un entorno configurado. No se deben presentar como ejecutados si solo se ejecutó el modo local.

## Referencias
- Portal Nacional de Datos Abiertos, conjunto **Vehículos involucrados en un accidente de tránsito Ley 2251-2022**, ID `6jmc-vaxk`.
- API Socrata: `https://www.datos.gov.co/resource/6jmc-vaxk.json`.
- Guía del proyecto de la asignatura, páginas 1–7.